# Short Tandem Repeats — SNP frequency by k-mer (official IWTomics + ggplot)

Mirrors the reference notebooks (`apr_snp_frequency_notebook_detected.ipynb`,
`str_snp_frequency_notebook_detected.ipynb`) but **stratified by repeat-unit length (k-mer)**
and comparing **detected vs undetected vs control**.

Pipeline (same upstream processing as the reference): annotation → labelled motif beds
(detected from `Short_Tandem_Repeat_detected.bed`, undetected = test ∖ detected) → ±100 bp
flanks → MR-style matched control (`bedtools shuffle`) → `locuschoice` → split
`LeftFlank|Locus|RightFlank` → per-chrom gnomAD intersect → per-k-mer matrices.
Significance = **official R `IWTomics`** (`IWTomicsTest`, B=1000, α=0.01), detected vs undetected,
run per k-mer. Plot = R + ggplot, `facet_grid(k-mer ~ region)`.

## 1. Build labelled motif beds (detected / undetected) + matched control
Carries the k-mer label (= Σ `composition`) on every locus; control inherits it from its source.

In [ ]:
"""STR SNP frequency by k-mer (paper-style IWTomics positional plot).

Mirrors the APR/STR notebook processing (split LeftFlank|Locus|RightFlank ->
flanks -> matched control -> locuschoice -> per-chrom gnomAD intersect ->
per-bin matrices -> IWTomics), but stratified by repeat-unit length (k-mer).

Layout: one row per k-mer (color = k-mer); each row = upstream(bp) | scaled Locus |
downstream(bp); detected = solid, undetected = dashed (same k-mer color),
control = grey dotted; grey shading = IWTomics(detected vs undetected) significant bins.
"""
import os, re, subprocess, random, csv as _csv
from collections import defaultdict
import numpy as np, pandas as pd

# config
BASE   = os.environ.get("SNP_AF_BASE", "/path/to/snp_af_analysis")
# Stage-1 (NR-Contrastive) outputs and experimental splits
# Override with environment variables; the defaults are the paths used for the paper.
PU_RESULTS      = os.environ.get("NONB_PU_RESULTS",
    "/path/to/pu_results")
EXPERIMENT_DATA = os.environ.get("NONB_EXPERIMENT_DATA",
    "/path/to/Experiment_Data")
IWTOMICS_DIR    = os.environ.get("IWTOMICS_DIR",
    "/path/to/nonb-nrcl-paper/snp_analysis/iwtomics")

WORK   = f"{BASE}/results/str_bykmer_run"
SNPDIR = f"{BASE}/snp_data/gnomad"
NCNR   = f"{BASE}/results/str_notebook_detected_run/NCNR.hg38.bed"
GAPS   = f"{BASE}/annotations/hg38_gaps.bed"
CHROMS = f"{BASE}/reference/hg38.chrom.sizes"
BT     = os.environ.get("BEDTOOLS", "bedtools")
TESTCSV= f"{EXPERIMENT_DATA}/Short_Tandem_Repeat_test.csv"
DETBED = f"{PU_RESULTS}/Short_Tandem_Repeat/Short_Tandem_Repeat_detected.bed"

MAX_FLANK  = 100
LOCUS_BINS = 180
FLANK      = 100          # bp of flank kept (we only plot/test 100bp)
MAXLEN     = 100          # motif length threshold
N_PERM     = 1000
ALPHA      = 0.01
KMERS      = [1, 2, 3, 4]
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
_csv.field_size_limit(10**7); random.seed(42)

scaledposLocus = [round(i,4) for i in np.arange(1, LOCUS_BINS*2, 2.0)/(LOCUS_BINS*2)]

def sh(cmd): subprocess.run(cmd, shell=True, check=True)

def locuschoice(in_bed, out_bed, seed=42):
    rng=random.Random(seed); last_chrom="chr0"; last_end=0; tochoose=[]; nkept=0
    with open(in_bed) as fi, open(out_bed,"w") as fo:
        for line in fi:
            arr=line.rstrip("\n").split("\t"); chrom=arr[0]; ws=int(arr[1]); we=int(arr[2])
            if chrom==last_chrom:
                if ws<last_end: tochoose.append(arr); last_end=we
                else:
                    fo.write("\t".join(tochoose[rng.randint(0,len(tochoose)-1)])+"\n"); nkept+=1
                    tochoose=[arr]; last_end=we
            else:
                if tochoose: fo.write("\t".join(tochoose[rng.randint(0,len(tochoose)-1)])+"\n"); nkept+=1
                tochoose=[arr]; last_chrom=chrom; last_end=we
        if tochoose: fo.write("\t".join(tochoose[rng.randint(0,len(tochoose)-1)])+"\n"); nkept+=1
    print(f"  locuschoice {in_bed} -> {out_bed}: kept {nkept}"); return nkept

def kmer_of(comp): return sum(int(re.match(r'(\d+)([ACGT])',t).group(1)) for t in comp.split('/'))

# 1. feature map + detected set
print("parsing features ...", flush=True)
feat={}
with open(TESTCSV) as fh:
    r=_csv.reader(fh); h=next(r)
    ci=h.index('chr'); li=h.index('label'); fi=h.index('features'); si=h.index('motif_start'); ei=h.index('motif_end')
    for row in r:
        if row[li]!='Short_Tandem_Repeat' or not row[si] or not row[ei]: continue
        d={}
        for p in row[fi].split(';'):
            if '=' in p: k,v=p.split('=',1); d[k]=v
        if 'composition' not in d: continue
        feat[(row[ci],int(float(row[si])),int(float(row[ei])))]=kmer_of(d['composition'])
det=set()
dd=pd.read_csv(DETBED,sep='\t').dropna(subset=['motif_start','motif_end'])
for c,s,e in zip(dd['chr'],dd['motif_start'].astype(int),dd['motif_end'].astype(int)): det.add((c,s,e))
print(f"  test STR loci {len(feat)}, detected {len(det)}", flush=True)

# 2. labeled motif beds (0-based, name=group|kmer), NCNR, <=100bp
with open("motif.det.bed","w") as fd, open("motif.und.bed","w") as fu, open("motif.all.bed","w") as fa:
    for (c,s1,e),k in feat.items():
        st=s1-1; L=e-st
        if L<=0 or L>MAXLEN: continue
        grp='detected' if (c,s1,e) in det else 'undetected'
        line=f"{c}\t{st}\t{e}\t{grp}|{k}\n"
        (fd if grp=='detected' else fu).write(line)
        fa.write(f"{c}\t{st}\t{e}\tall|{k}\n")
for nm in ("det","und","all"):
    sh(f"{BT} intersect -a motif.{nm}.bed -b {NCNR} -v | sort -k1,1 -k2,2n > motif.{nm}.ncnr.bed")

# 3. matched control = shuffle of all test STR (inherits kmer)
sh(f"cut -f1-3 motif.all.bed | sort -k1,1 -k2,2n > str_all_coords.bed")
sh(f"cat str_all_coords.bed {GAPS} {NCNR} | cut -f1-3 | sort -k1,1 -k2,2n | {BT} merge -i stdin > exclude.bed")
sh(f"{BT} shuffle -i motif.all.ncnr.bed -excl exclude.bed -g {CHROMS} -chrom -f 0 -noOverlapping -seed 42 "
   "| awk 'BEGIN{OFS=\"\\t\"}{split($4,a,\"|\"); print $1,$2,$3,\"control|\"a[2]}' "
   "| sort -k1,1 -k2,2n > motif.ctrl.ncnr.bed")

# 4. flank + locuschoice + split (record locus_id -> kmer)
def flank_choose_split(motif_ncnr, tag, seed):
    # build flanked: chrom f_start f_end name l_start l_end   (clamp f_start at 0)
    with open(motif_ncnr) as fi, open(f"{tag}.flanked.bed","w") as fo:
        for line in fi:
            c,s,e,name=line.rstrip("\n").split("\t")
            s=int(s); e=int(e); fs=max(0,s-FLANK); fe=e+FLANK
            fo.write(f"{c}\t{fs}\t{fe}\t{name}\t{s}\t{e}\n")
    sh(f"sort -k1,1 -k2,2n {tag}.flanked.bed > {tag}.flanked.sorted.bed")
    locuschoice(f"{tag}.flanked.sorted.bed", f"{tag}.nooverlap.bed", seed=seed)
    kmap=[]
    with open(f"{tag}.nooverlap.bed") as fi, open(f"{tag}.split.bed","w") as fo:
        for line in fi:
            c,fs,fe,name,ls,le=line.rstrip("\n").split("\t")
            lid=len(kmap); kmap.append(int(name.split("|")[1]))
            fo.write(f"{c}\t{fs}\t{ls}\t{lid}\tLeftFlank\n")
            fo.write(f"{c}\t{ls}\t{le}\t{lid}\tLocus\n")
            fo.write(f"{c}\t{le}\t{fe}\t{lid}\tRightFlank\n")
    sh(f"sort -k1,1 -k2,2n {tag}.split.bed > {tag}.split.sorted.bed && mv {tag}.split.sorted.bed {tag}.split.bed")
    return np.array(kmap, dtype=int)

print("flank/choose/split ...", flush=True)
kmap_det = flank_choose_split("motif.det.ncnr.bed",  "det",  seed=42)
kmap_und = flank_choose_split("motif.und.ncnr.bed",  "und",  seed=42)
kmap_ctl = flank_choose_split("motif.ctrl.ncnr.bed", "ctrl", seed=43)
print(f"  loci det/und/ctrl = {len(kmap_det)}/{len(kmap_und)}/{len(kmap_ctl)}", flush=True)

# 5. per-chrom gnomAD intersect (-loj)
def intersect(tag):
    chroms=sorted(set(l.split("\t")[0] for l in open(f"{tag}.split.bed")))
    with open(f"{tag}.intersect","w") as fo:
        for c in chroms:
            snp=f"{SNPDIR}/{c}_snps.bed"
            if not os.path.exists(snp): continue
            cmd=f"awk '$1==\"{c}\"' {tag}.split.bed | {BT} intersect -wa -wb -a stdin -b {snp} -loj"
            fo.write(subprocess.run(cmd,shell=True,capture_output=True,text=True).stdout)
    print(f"  intersect {tag} done", flush=True)
intersect("det"); intersect("und"); intersect("ctrl")

## 2. Per-k-mer matrices + per-bin frequency table
Builds the integer count matrices (loci × bins) and the tidy per-bin frequency CSV
(`str_bykmer_freq.csv`) consumed by the plot.

In [ ]:
"""STR-by-kmer preparation, stratified by k-mer; mirrors the Python half of the
STR notebook's IWTomics section.
Reuses the per-kmer intersects already in results/str_bykmer_run/.
Outputs (in str_bykmer_run/):
  str_bykmer_freq.csv               tidy per-bin freq: region,kmer,x,detected,undetected,control
  mat_{det,und}_k{K}_{L,C,R}.txt.gz integer count matrices (loci x bins) for official R IWTomics
"""
import os, numpy as np, pandas as pd

WORK=os.path.join(os.environ.get("SNP_AF_BASE", "/path/to/snp_af_analysis"), "results", "str_bykmer_run")
os.chdir(WORK)
MAX_FLANK=100; LOCUS_BINS=180; KMERS=[1,2,3,4]
scaledposLocus=[round(i,4) for i in np.arange(1,LOCUS_BINS*2,2.0)/(LOCUS_BINS*2)]

def kmap_from_bed(bed):
    return np.array([int(l.split("\t")[3].split("|")[1]) for l in open(bed)], dtype=int)

def build_matrices(intersect_path, n_loci):
    L=np.zeros((n_loci,MAX_FLANK),np.int32); C=np.zeros((n_loci,LOCUS_BINS),np.int32); R=np.zeros((n_loci,MAX_FLANK),np.int32)
    seen=set()
    with open(intersect_path) as fh:
        for line in fh:
            f=line.rstrip("\n").split("\t")
            if len(f)<8: continue
            if len(f)>=6 and f[5]==".": continue
            lid=int(f[3]); ps=int(f[1]); pe=int(f[2]); ptype=f[4]; snp=int(f[6])
            key=(lid,ptype,snp)
            if key in seen: continue
            seen.add(key); plen=pe-ps
            if ptype=="LeftFlank":
                idx=-(pe-snp)+MAX_FLANK
                if 0<=idx<MAX_FLANK: L[lid,idx]+=1
            elif ptype=="RightFlank":
                idx=snp-ps
                if 0<=idx<MAX_FLANK: R[lid,idx]+=1
            elif ptype=="Locus" and plen>0:
                fsf=(snp-ps)/plen; fef=(snp+1-ps)/plen
                for i,sp in enumerate(scaledposLocus):
                    if fsf<=sp<fef: C[lid,i]+=1
    return {"L":L,"C":C,"R":R}

print("kmer maps ...",flush=True)
km={"det":kmap_from_bed("det.nooverlap.bed"),
    "und":kmap_from_bed("und.nooverlap.bed"),
    "ctrl":kmap_from_bed("ctrl.nooverlap.bed")}
print("build matrices ...",flush=True)
mats={"det":build_matrices("det.intersect",len(km["det"])),
      "und":build_matrices("und.intersect",len(km["und"])),
      "ctrl":build_matrices("ctrl.intersect",len(km["ctrl"]))}

REG_X={"L":np.arange(-MAX_FLANK,0),"C":np.arange(1,LOCUS_BINS+1),"R":np.arange(1,MAX_FLANK+1)}
rows=[]
for k in KMERS:
    for reg in ("L","C","R"):
        d=mats["det"][reg][km["det"]==k]; u=mats["und"][reg][km["und"]==k]; c=mats["ctrl"][reg][km["ctrl"]==k]
        df=mats["det"][reg].shape[1]
        det_f=d.sum(0)/len(d) if len(d) else np.full(df,np.nan)
        und_f=u.sum(0)/len(u) if len(u) else np.full(df,np.nan)
        ctl_f=c.sum(0)/len(c) if len(c) else np.full(df,np.nan)
        for x,a,b,cc in zip(REG_X[reg],det_f,und_f,ctl_f):
            rows.append((reg,k,int(x),a,b,cc))
        # matrices for official R IWTomics (det vs undet)
        np.savetxt(f"mat_det_k{k}_{reg}.txt.gz", d, fmt="%d")
        np.savetxt(f"mat_und_k{k}_{reg}.txt.gz", u, fmt="%d")
        print(f"  k{k} {reg}: det {len(d)}, und {len(u)}, ctrl {len(c)} loci",flush=True)

pd.DataFrame(rows,columns=["region","kmer","x","detected","undetected","control"]).to_csv("str_bykmer_freq.csv",index=False)
print("wrote str_bykmer_freq.csv + mat_*.txt.gz",flush=True)

## 3. Significance — official R `IWTomics` (detected vs undetected, per k-mer)
Runs `iwtomics_str/run_iwtomics_str_bykmer_dvu.R` → `str_bykmer_iwt.csv`
(region, k-mer, x, adj_pvalue, significant).

In [ ]:
import subprocess
R_BIN = os.environ.get("R_BIN", "$R_BIN")
proc = subprocess.run([R_BIN, "--vanilla",
                       f"{IWTOMICS_DIR}/str/run_iwtomics_str_bykmer_dvu.R"],
                      capture_output=True, text=True)
print(proc.stdout[-2000:])
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-3000:]); raise RuntimeError("IWTomics R failed")

## 4. Plot — R + ggplot, faceted by k-mer
detected = solid, undetected = dashed (colour = k-mer), control = grey points,
grey vertical bands = IWTomics(det vs undet) significant bins.

In [ ]:
import subprocess
from IPython.display import Image, display
R_BIN = os.environ.get("R_BIN", "$R_BIN")
proc = subprocess.run([R_BIN, "--vanilla",
                       f"{IWTOMICS_DIR}/str/plot_str_bykmer.R"],
                      capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-3000:]); raise RuntimeError("ggplot R failed")
display(Image("/path/to/snp_af_analysis/results/str_bykmer_run/str_snp_frequency_by_kmer_iwtomics.png"))